# 13–14. Bağımsız Test Değerlendirmesi ve Zamansal Analiz

**Ön koşul:** `main.ipynb` çalıştırılmış ve `final_pipeline.json`, `test_predictions_locked.csv`, `test_lock.json` dosyaları **ayrı bir commit** ile GitHub'a yüklenmiş olmalıdır.

Bu notebook:
1. Kilit dosyalarının SHA-256 özetlerini doğrular (tahminler sonradan değiştirilmemiş olmalı).
2. Test referans sayılarını **ilk kez** `02_GT/TRA` klasöründen okur.
3. Tablo 4 (kare bazında sonuçlar), Tablo 5 (özet metrikler), Şekil 5 (zamansal grafik) ve Şekil 6 (başarılı/hatalı örnekler) üretir.

Tahminler **yeniden hesaplanmaz**; değerlendirme yalnızca kilitli CSV üzerinden yapılır. Şekil 6 için kilitli konfigürasyon görselleştirme amacıyla yeniden çalıştırılır ve sayının kilitli değerle aynı olduğu doğrulanır.

In [ ]:
import sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd(); sys.path.insert(0, str(PROJECT_DIR))
from cellcount.io_utils import ensure_dataset, load_image, reference_count, read_tif, tra_path
from cellcount.pipeline import run_pipeline, describe, load_config
from cellcount.metrics import count_metrics
from cellcount import viz

ROOT = ensure_dataset(PROJECT_DIR / "data")
FIG_DIR = PROJECT_DIR / "figures"; PRIV_DIR = FIG_DIR / "private"; PRIV_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200, "font.size": 9})

lock = json.load(open("test_lock.json"))
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for fn in ["final_pipeline.json", "test_predictions_locked.csv"]:
    assert sha(fn) == lock[fn], f"{fn} kilitlendikten sonra DEĞİŞTİRİLMİŞ!"
print("Kilit doğrulandı:", lock["locked_at"])

cfg_all = load_config("final_pipeline.json")
FINAL, BASELINE = cfg_all["pipeline"], cfg_all["baseline"]
HM_REF = load_image(ROOT, "01", cfg_all["histogram_matching_reference"]["frame"]) if cfg_all["histogram_matching_reference"]["used"] else None
locked = pd.read_csv("test_predictions_locked.csv")
print("Nihai:", describe(FINAL)); print("Baseline:", describe(BASELINE))

## 13. Test referanslarının okunması ve Tablo 4

In [ ]:
res = locked.copy()
res["ref_count"] = [reference_count(ROOT, "02", int(f)) for f in res.frame]
assert res.ref_count.notna().all(), "Referansı olmayan test karesi var"
res["baseline_error"] = res.baseline_pred - res.ref_count
res["final_error"] = res.final_pred - res.ref_count
res["baseline_abs_error"] = res.baseline_error.abs()
res["final_abs_error"] = res.final_error.abs()
res.to_csv("test_results.csv", index=False)
table4 = res[["test_block", "frame", "ref_count", "baseline_pred", "final_pred", "baseline_abs_error", "final_abs_error"]]
table4.columns = ["Blok", "Kare", "Gerçek sayı", "Baseline tahmini", "Nihai tahmin", "Baseline mutlak hata", "Nihai mutlak hata"]
table4

## Tablo 5 – Özet test metrikleri (ve doğrulama ile karşılaştırma)

In [ ]:
summary = []
for name, p, t in (("Baseline", "baseline_pred", "baseline_time_ms"), ("Nihai pipeline", "final_pred", "final_time_ms")):
    m = count_metrics(res[p], res.ref_count, res[t])
    summary.append({"Yöntem": name, "MAE": m["MAE"], "RMSE": m["RMSE"], "Bias": m["Bias"],
                    "Exact-count Accuracy": m["ExactAcc"], "Ortalama süre (ms)": m["mean_time_s"]})
table5 = pd.DataFrame(summary)
table5.to_csv("test_summary.csv", index=False)

per_block = []
for b, g in res.groupby("test_block"):
    for name, p in (("Baseline", "baseline_pred"), ("Nihai pipeline", "final_pred")):
        m = count_metrics(g[p], g.ref_count)
        per_block.append({"Blok": b, "Kareler": f"{g.frame.min()}-{g.frame.max()}", "Yöntem": name, **{k: m[k] for k in ["MAE", "RMSE", "Bias", "ExactAcc"]}})
per_block = pd.DataFrame(per_block); per_block.to_csv("test_per_block.csv", index=False)

val = pd.read_csv("validation_results.csv")
cmp = pd.DataFrame({"Küme": ["Doğrulama (01)", "Test (02)"],
                    "Baseline MAE": [val.loc[~val.is_candidate, "val_MAE"].iloc[0], table5.MAE[0]],
                    "Nihai MAE": [val.loc[val.selected, "val_MAE"].iloc[0], table5.MAE[1]],
                    "Nihai RMSE": [val.loc[val.selected, "val_RMSE"].iloc[0], table5.RMSE[1]],
                    "Nihai Bias": [val.loc[val.selected, "val_Bias"].iloc[0], table5.Bias[1]]})
display(table5.round(3)); display(per_block.round(3)); display(cmp.round(3))

## 14. Şekil 5 – Zamansal hücre sayısı analizi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (b, g) in zip(axes, res.groupby("test_block")):
    ax.plot(g.frame, g.ref_count, "k-o", label="Gerçek (TRA)")
    ax.plot(g.frame, g.baseline_pred, "--s", color="tab:red", label="Baseline")
    ax.plot(g.frame, g.final_pred, "-^", color="tab:green", label="Nihai pipeline")
    ax.set_title(f"Test bloğu {b} (Seq 02, t{g.frame.min():03d}–t{g.frame.max():03d})"); ax.set_xlabel("Kare"); ax.set_xticks(g.frame); ax.grid(alpha=.3)
axes[0].set_ylabel("Hücre sayısı"); axes[0].legend()
plt.suptitle("Şekil 5. Üç test bloğunda gerçek ve tahmin edilen hücre sayılarının zamansal değişimi")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig5_temporal_counts.png", bbox_inches="tight"); plt.show()

temporal = []
for b, g in res.groupby("test_block"):
    g = g.sort_values("frame")
    for col in ["ref_count", "baseline_pred", "final_pred"]:
        d = g[col].diff().dropna()
        temporal.append({"Blok": b, "Seri": col, "Ortalama |Δ|": d.abs().mean(), "Maks |Δ|": d.abs().max(), "Std": g[col].std()})
temporal = pd.DataFrame(temporal); temporal.to_csv("test_temporal_stats.csv", index=False)
temporal.round(3)

*Yorum rehberi:* Gerçek sayıdaki ardışık değişimler (bölünme, görüş alanına giriş/çıkış) ile tahmindeki değişimler karşılaştırılmalıdır. Gerçek sayı sabitken tahminde görülen ±1–2'lik sıçramalar segmentasyon kararsızlığına (ör. soluk bir hücrenin eşik etrafında gidip gelmesi, temas eden hücrelerin bazı karelerde ayrışıp bazılarında birleşmesi) işaret eder.

## Şekil 6 – Başarılı, hatalı ve zorlayıcı sayım örnekleri

In [ ]:
r_sorted = res.sort_values(["final_abs_error", "frame"])
picks = [("Başarılı", r_sorted.iloc[0]), ("En hatalı", r_sorted.iloc[-1]),
         ("Zorlayıcı (baseline'ın en kötü olduğu)", res.sort_values("baseline_abs_error").iloc[-1])]
db, dfn = describe(BASELINE), describe(FINAL)
base_lbl = f"Baseline [{db['segmentation']}, CCL, min_area={db['min_area']}]"
final_lbl = f"Nihai [{dfn['intensity']}→{dfn['filter']}→{dfn['segmentation']}→{dfn['morphology']}→WS md={FINAL['separation'].get('min_distance')}, min_area={dfn['min_area']}]"
fig, ax = plt.subplots(len(picks), 3, figsize=(16, 5.6 * len(picks)))
for i, (tag, r) in enumerate(picks):
    f = int(r.frame); im = load_image(ROOT, "02", f); tra = read_tif(tra_path(ROOT, "02", f))
    nb, lb, _ = run_pipeline(im, BASELINE); nf, lf, _ = run_pipeline(im, FINAL, HM_REF)
    assert nb == r.baseline_pred and nf == r.final_pred, "Görselleştirme kilitli tahminle uyuşmuyor"
    ax[i, 0].imshow(viz.display_gain(im), cmap="gray"); viz.overlay_markers(ax[i, 0], tra); ax[i, 0].set_title(f"{tag}: t{f:03d} – gerçek {int(r.ref_count)} (+: TRA)", fontsize=9)
    ax[i, 1].imshow(viz.overlay_labels(im, lb)); viz.overlay_markers(ax[i, 1], tra, "cyan"); ax[i, 1].set_title(f"{base_lbl}\nsayı {nb} (hata {nb - int(r.ref_count):+d})", fontsize=8)
    ax[i, 2].imshow(viz.overlay_labels(im, lf)); viz.overlay_markers(ax[i, 2], tra, "cyan"); ax[i, 2].set_title(f"{final_lbl}\nsayı {nf} (hata {nf - int(r.ref_count):+d})", fontsize=7)
for a in ax.ravel(): a.axis("off")
plt.suptitle("Şekil 6. Başarılı, hatalı ve zorlayıcı sayım örnekleri (Sequence 02)", y=1.01)
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig6_success_failure_examples.png", bbox_inches="tight"); plt.show()

In [ ]:
# Hata analizi: hangi TRA hücreleri kaçırıldı / hangi tespitler referansla eşleşmedi?
# nonzero_frac: hücre işaretçisi içinde ham değeri 0'dan büyük piksellerin oranı. Ham görüntü düşük seviyelerde
# kuantize olduğundan (ör. 0, 2/3, 5/6, ...) bu oran, soluk bir çekirdeğin arka plandan ne kadar ayrışabildiğinin doğrudan ölçüsüdür.
rows, cell_rows = [], []
for _, r in res.iterrows():
    f = int(r.frame); im = load_image(ROOT, "02", f); tra = read_tif(tra_path(ROOT, "02", f))
    _, lf, _ = run_pipeline(im, FINAL, HM_REF)
    ids = np.unique(tra)[1:]
    for lab in ids:
        m = tra == lab
        cell_rows.append(dict(frame=f, cell=int(lab), detected=bool((lf[m] > 0).any()),
                              median_intensity=float(np.median(im[m])), nonzero_frac=float((im[m] > 0).mean())))
    hit = {lab: np.unique(lf[tra == lab]) for lab in ids}
    missed = [lab for lab, v in hit.items() if np.all(v == 0)]
    det_hits = {d: np.unique(tra[lf == d]) for d in np.unique(lf)[1:]}
    rows.append(dict(frame=f, ref=int(r.ref_count), pred=int(r.final_pred), missed_cells=len(missed),
                     unmatched_detections=sum(1 for v in det_hits.values() if np.all(v == 0)),
                     merged_detections=sum(1 for v in det_hits.values() if np.count_nonzero(v) >= 2),
                     missed_median_intensity=float(np.median([np.median(im[tra == l]) for l in missed])) if missed else np.nan))
err = pd.DataFrame(rows); err.to_csv("test_error_analysis.csv", index=False)
cells_df = pd.DataFrame(cell_rows); cells_df.to_csv("test_cell_detection.csv", index=False)
display(err)
cells_df.groupby("detected")[["median_intensity", "nonzero_frac"]].describe().round(2)